# Creating Database "EuroEatsDB"

In [24]:
import pyodbc
import pandas as pd

In [25]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};" # Use print(pyodbc.drivers())
    "SERVER=DESKTOP-0SIEUH6\SQLEXPRESS;"
    "DATABASE=master;"
    "Trusted_Connection=yes;",
    autocommit=True)

cursor = conn.cursor()

cursor.execute("create database EuroEatsDB;")

cursor.close()

conn.close()

# Using "EuroEatsDB"

In [26]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-0SIEUH6\\SQLEXPRESS;"
    "DATABASE=EuroEatsDB;"   # <-- note: EuroEatsDB, not master
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()

# Importing CSV's

In [27]:
customers = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\EuroEats\dataset\customers.csv")
deliveries = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\EuroEats\dataset\deliveries.csv")
drivers = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\EuroEats\dataset\drivers.csv")
menu_items = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\EuroEats\dataset\menu_items.csv")
order_items = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\EuroEats\dataset\order_items.csv")
orders = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\EuroEats\dataset\orders.csv")
payments = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\EuroEats\dataset\payments.csv")
restaurants = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\EuroEats\dataset\restaurants.csv")


#
###### =================================================================================================================================
# EUROEATS DATABASE TABLE CREATION
###### =================================================================================================================================

In [28]:
# ============================================================
# EUROEATS — CREATE ALL TABLES
# ============================================================

# ------------------------------------------------------------
# 1. CUSTOMERS
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE dbo.customers (
    customer_id       VARCHAR(50)  NOT NULL PRIMARY KEY,
    first_name        VARCHAR(100),
    last_name         VARCHAR(100),
    age               VARCHAR(20),
    gender            VARCHAR(20),
    country           VARCHAR(100),
    city              VARCHAR(100),
    registration_date VARCHAR(50),
    customer_segment  VARCHAR(30),
    loyalty_status    VARCHAR(30)
);
""")


# ------------------------------------------------------------
# 2. RESTAURANTS
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE dbo.restaurants (
    restaurant_id            VARCHAR(50)  NOT NULL PRIMARY KEY,
    restaurant_name          VARCHAR(150),
    country                  VARCHAR(100),
    city                     VARCHAR(100),
    cuisine_type             VARCHAR(50),
    price_level              VARCHAR(20),
    restaurant_rating        VARCHAR(20),
    average_preparation_time VARCHAR(30),
    restaurant_join_date     VARCHAR(50),
    restaurant_size          VARCHAR(30),
    average_daily_orders     VARCHAR(30)
);
""")


# ------------------------------------------------------------
# 3. MENU ITEMS
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE dbo.menu_items (
    item_id                  VARCHAR(50)  NOT NULL PRIMARY KEY,
    restaurant_id            VARCHAR(50)  NOT NULL,
    item_name                VARCHAR(150),
    category                 VARCHAR(30),
    price_eur                VARCHAR(30),
    preparation_time_minutes VARCHAR(30),
    calories                 VARCHAR(30),
    vegetarian               VARCHAR(20),
    popularity_score         VARCHAR(30),

    CONSTRAINT FK_menu_items_restaurants
        FOREIGN KEY (restaurant_id)
        REFERENCES dbo.restaurants(restaurant_id)
);
""")


# ------------------------------------------------------------
# 4. DRIVERS
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE dbo.drivers (
    driver_id             VARCHAR(50)  NOT NULL PRIMARY KEY,
    first_name            VARCHAR(100),
    last_name             VARCHAR(100),
    country               VARCHAR(100),
    city                  VARCHAR(100),
    vehicle_type          VARCHAR(30),
    years_experience      VARCHAR(30),
    driver_rating         VARCHAR(30),
    completed_deliveries  VARCHAR(30),
    employment_type       VARCHAR(30),
    average_speed_kmh     VARCHAR(30)
);
""")


# ------------------------------------------------------------
# 5. ORDERS
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE dbo.orders (
    order_id          VARCHAR(50)  NOT NULL PRIMARY KEY,
    customer_id       VARCHAR(50)  NOT NULL,
    restaurant_id     VARCHAR(50)  NOT NULL,
    order_datetime    VARCHAR(50),
    number_of_items   VARCHAR(30),
    subtotal_eur      VARCHAR(30),
    delivery_fee_eur  VARCHAR(30),
    discount_eur      VARCHAR(30),
    total_amount_eur  VARCHAR(30),
    payment_method    VARCHAR(30),
    order_status      VARCHAR(30),

    CONSTRAINT FK_orders_customers
        FOREIGN KEY (customer_id)
        REFERENCES dbo.customers(customer_id),

    CONSTRAINT FK_orders_restaurants
        FOREIGN KEY (restaurant_id)
        REFERENCES dbo.restaurants(restaurant_id)
);
""")


# ------------------------------------------------------------
# 6. ORDER ITEMS
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE dbo.order_items (
    order_item_id      VARCHAR(50)  NOT NULL PRIMARY KEY,
    order_id           VARCHAR(50)  NOT NULL,
    item_id            VARCHAR(50)  NOT NULL,
    quantity           VARCHAR(30),
    unit_price_eur     VARCHAR(30),
    item_discount_eur  VARCHAR(30),

    CONSTRAINT FK_order_items_orders
        FOREIGN KEY (order_id)
        REFERENCES dbo.orders(order_id),

    CONSTRAINT FK_order_items_menu_items
        FOREIGN KEY (item_id)
        REFERENCES dbo.menu_items(item_id)
);
""")


# ------------------------------------------------------------
# 7. DELIVERIES
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE dbo.deliveries (
    delivery_id                VARCHAR(50)  NOT NULL PRIMARY KEY,
    order_id                   VARCHAR(50)  NOT NULL UNIQUE,
    driver_id                  VARCHAR(50)  NOT NULL,
    distance_km                VARCHAR(30),
    traffic_level              VARCHAR(30),
    weather_condition          VARCHAR(30),
    temperature_c              VARCHAR(30),
    pickup_delay_minutes       VARCHAR(30),
    restaurant_wait_minutes    VARCHAR(30),
    estimated_delivery_minutes VARCHAR(30),
    actual_delivery_minutes    VARCHAR(30),
    delivery_status            VARCHAR(30),

    CONSTRAINT FK_deliveries_orders
        FOREIGN KEY (order_id)
        REFERENCES dbo.orders(order_id),

    CONSTRAINT FK_deliveries_drivers
        FOREIGN KEY (driver_id)
        REFERENCES dbo.drivers(driver_id)
);
""")


# ------------------------------------------------------------
# 8. PAYMENTS
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE dbo.payments (
    payment_id       VARCHAR(50)  NOT NULL PRIMARY KEY,
    order_id         VARCHAR(50)  NOT NULL UNIQUE,
    payment_datetime VARCHAR(50),
    payment_method   VARCHAR(30),
    amount_eur       VARCHAR(30),
    payment_status   VARCHAR(30),
    currency         VARCHAR(10),

    CONSTRAINT FK_payments_orders
        FOREIGN KEY (order_id)
        REFERENCES dbo.orders(order_id)
);
""")


# ------------------------------------------------------------
# 9. REVIEWS
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE dbo.reviews (
    review_id        VARCHAR(50)  NOT NULL PRIMARY KEY,
    order_id         VARCHAR(50)  NOT NULL UNIQUE,
    customer_id      VARCHAR(50)  NOT NULL,
    rating           VARCHAR(20),
    review_datetime  VARCHAR(50),
    delivery_rating  VARCHAR(20),
    food_rating      VARCHAR(20),
    review_sentiment VARCHAR(30),

    CONSTRAINT FK_reviews_orders
        FOREIGN KEY (order_id)
        REFERENCES dbo.orders(order_id),

    CONSTRAINT FK_reviews_customers
        FOREIGN KEY (customer_id)
        REFERENCES dbo.customers(customer_id)
);
""")


conn.commit()

print("All 9 EuroEats tables created successfully.")

All 9 EuroEats tables created successfully.


In [29]:
# ============================================================
# EUROEATS — INSERT ALL DATA
# ============================================================

cursor.fast_executemany = False


# ------------------------------------------------------------
# 1. CUSTOMERS
# ------------------------------------------------------------

customers_data = list(
    customers[
        [
            "customer_id",
            "first_name",
            "last_name",
            "age",
            "gender",
            "country",
            "city",
            "registration_date",
            "customer_segment",
            "loyalty_status"
        ]
    ].itertuples(index=False, name=None)
)

cursor.executemany("""
    INSERT INTO dbo.customers
    (
        customer_id,
        first_name,
        last_name,
        age,
        gender,
        country,
        city,
        registration_date,
        customer_segment,
        loyalty_status
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", customers_data)

conn.commit()

print(f"customers       : {len(customers_data):,} rows inserted")


# ------------------------------------------------------------
# 2. RESTAURANTS
# ------------------------------------------------------------

restaurants_data = list(
    restaurants[
        [
            "restaurant_id",
            "restaurant_name",
            "country",
            "city",
            "cuisine_type",
            "price_level",
            "restaurant_rating",
            "average_preparation_time",
            "restaurant_join_date",
            "restaurant_size",
            "average_daily_orders"
        ]
    ].itertuples(index=False, name=None)
)

cursor.executemany("""
    INSERT INTO dbo.restaurants
    (
        restaurant_id,
        restaurant_name,
        country,
        city,
        cuisine_type,
        price_level,
        restaurant_rating,
        average_preparation_time,
        restaurant_join_date,
        restaurant_size,
        average_daily_orders
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", restaurants_data)

conn.commit()

print(f"restaurants     : {len(restaurants_data):,} rows inserted")


# ------------------------------------------------------------
# 3. MENU ITEMS
# ------------------------------------------------------------

menu_items_data = list(
    menu_items[
        [
            "item_id",
            "restaurant_id",
            "item_name",
            "category",
            "price_eur",
            "preparation_time_minutes",
            "calories",
            "vegetarian",
            "popularity_score"
        ]
    ].itertuples(index=False, name=None)
)

cursor.executemany("""
    INSERT INTO dbo.menu_items
    (
        item_id,
        restaurant_id,
        item_name,
        category,
        price_eur,
        preparation_time_minutes,
        calories,
        vegetarian,
        popularity_score
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""", menu_items_data)

conn.commit()

print(f"menu_items      : {len(menu_items_data):,} rows inserted")


# ------------------------------------------------------------
# 4. DRIVERS
# ------------------------------------------------------------

drivers_data = list(
    drivers[
        [
            "driver_id",
            "first_name",
            "last_name",
            "country",
            "city",
            "vehicle_type",
            "years_experience",
            "driver_rating",
            "completed_deliveries",
            "employment_type",
            "average_speed_kmh"
        ]
    ].itertuples(index=False, name=None)
)

cursor.executemany("""
    INSERT INTO dbo.drivers
    (
        driver_id,
        first_name,
        last_name,
        country,
        city,
        vehicle_type,
        years_experience,
        driver_rating,
        completed_deliveries,
        employment_type,
        average_speed_kmh
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", drivers_data)

conn.commit()

print(f"drivers         : {len(drivers_data):,} rows inserted")


# ------------------------------------------------------------
# 5. ORDERS
# ------------------------------------------------------------

orders_data = list(
    orders[
        [
            "order_id",
            "customer_id",
            "restaurant_id",
            "order_datetime",
            "number_of_items",
            "subtotal_eur",
            "delivery_fee_eur",
            "discount_eur",
            "total_amount_eur",
            "payment_method",
            "order_status"
        ]
    ].itertuples(index=False, name=None)
)

cursor.executemany("""
    INSERT INTO dbo.orders
    (
        order_id,
        customer_id,
        restaurant_id,
        order_datetime,
        number_of_items,
        subtotal_eur,
        delivery_fee_eur,
        discount_eur,
        total_amount_eur,
        payment_method,
        order_status
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", orders_data)

conn.commit()

print(f"orders          : {len(orders_data):,} rows inserted")


# ------------------------------------------------------------
# 6. ORDER ITEMS
# ------------------------------------------------------------

order_items_data = list(
    order_items[
        [
            "order_item_id",
            "order_id",
            "item_id",
            "quantity",
            "unit_price_eur",
            "item_discount_eur"
        ]
    ].itertuples(index=False, name=None)
)

cursor.executemany("""
    INSERT INTO dbo.order_items
    (
        order_item_id,
        order_id,
        item_id,
        quantity,
        unit_price_eur,
        item_discount_eur
    )
    VALUES (?, ?, ?, ?, ?, ?)
""", order_items_data)

conn.commit()

print(f"order_items     : {len(order_items_data):,} rows inserted")


# ------------------------------------------------------------
# 7. DELIVERIES
# ------------------------------------------------------------

deliveries_data = list(
    deliveries[
        [
            "delivery_id",
            "order_id",
            "driver_id",
            "distance_km",
            "traffic_level",
            "weather_condition",
            "temperature_c",
            "pickup_delay_minutes",
            "restaurant_wait_minutes",
            "estimated_delivery_minutes",
            "actual_delivery_minutes",
            "delivery_status"
        ]
    ].itertuples(index=False, name=None)
)

cursor.executemany("""
    INSERT INTO dbo.deliveries
    (
        delivery_id,
        order_id,
        driver_id,
        distance_km,
        traffic_level,
        weather_condition,
        temperature_c,
        pickup_delay_minutes,
        restaurant_wait_minutes,
        estimated_delivery_minutes,
        actual_delivery_minutes,
        delivery_status
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", deliveries_data)

conn.commit()

print(f"deliveries      : {len(deliveries_data):,} rows inserted")


# ------------------------------------------------------------
# 8. PAYMENTS
# ------------------------------------------------------------

payments_data = list(
    payments[
        [
            "payment_id",
            "order_id",
            "payment_datetime",
            "payment_method",
            "amount_eur",
            "payment_status",
            "currency"
        ]
    ].itertuples(index=False, name=None)
)

cursor.executemany("""
    INSERT INTO dbo.payments
    (
        payment_id,
        order_id,
        payment_datetime,
        payment_method,
        amount_eur,
        payment_status,
        currency
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
""", payments_data)

conn.commit()

print(f"payments        : {len(payments_data):,} rows inserted")


# ------------------------------------------------------------
# 9. REVIEWS
# ------------------------------------------------------------

reviews_data = list(
    reviews[
        [
            "review_id",
            "order_id",
            "customer_id",
            "rating",
            "review_datetime",
            "delivery_rating",
            "food_rating",
            "review_sentiment"
        ]
    ].itertuples(index=False, name=None)
)

cursor.executemany("""
    INSERT INTO dbo.reviews
    (
        review_id,
        order_id,
        customer_id,
        rating,
        review_datetime,
        delivery_rating,
        food_rating,
        review_sentiment
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", reviews_data)

conn.commit()

print(f"reviews         : {len(reviews_data):,} rows inserted")


# ============================================================
# FINAL CHECK
# ============================================================

print("\nAll EuroEats data inserted successfully.")

customers       : 1,500 rows inserted
restaurants     : 120 rows inserted
menu_items      : 1,500 rows inserted
drivers         : 350 rows inserted
orders          : 4,500 rows inserted
order_items     : 4,697 rows inserted
deliveries      : 1,000 rows inserted
payments        : 800 rows inserted
reviews         : 500 rows inserted

All EuroEats data inserted successfully.


# ---------------------------- Database Created sucessfully ----------------------------